# Faruq-v3 AF2 vs AF2+FFAB2 — Kaggle from-start paired run
Frozen fair comparison for one declared seed. Both arms start from the same seed-matched D0 and train for the same 50 epochs.

**Required Kaggle input:** existing private dataset `faruq-v3-experiment-core-v1`.

Run this notebook separately for seed 42, 123, and 2026. It evaluates grouped development `val` only. Locked test, DCT training before the three-seed Stage-1 decision, and post-result retuning are forbidden here.


In [ ]:
SEED=42  # rerun as a separate Kaggle version/runtime with 123 or 2026
assert SEED in (42,123,2026)
BRANCH='codex/af2-ffab2-from-start-dct'


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
core=sorted(INPUT.rglob('af2_spectral_kaggle_manifest.json'))
if len(core)!=1: raise FileNotFoundError(f'Harus ada tepat satu core manifest; ditemukan {core}')
print('INPUT PREFLIGHT PASS')
print('CORE:',core[0])


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,torch
os.chdir(WORK)
REPO=WORK/'coffee-bean-detection'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All.')


In [ ]:
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
DATA,ARTIFACTS,CORE_CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
assert CORE_CONTRACT['test_images_accessed'] is False
D0=Path(ARTIFACTS[f'D0_seed{SEED}_best.pt'])
GROUPED=DATA/'faruq_grouped_summary.json'
if not D0.is_file() or not GROUPED.is_file(): raise FileNotFoundError(f'Core artifact missing: D0={D0} GROUPED={GROUPED}')
OUT=WORK/f'af2-ffab2-from-start-seed{SEED}-v1'; OUT.mkdir(exist_ok=True)
STATIC=OUT/'val_reports'/f'from_start_static_audit_seed{SEED}.json'
print('CORE CONTRACT PASS')
print('DATA:',DATA)
print('D0:',D0)
print('GROUPED:',GROUPED)
print('OUTPUT:',OUT)


In [ ]:
# Contract tests + static authorization before either arm trains.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_af2_ffa.py','tests/test_af2_ffa_from_start_dct.py','tests/test_af2_ffa_from_start_decision.py'],cwd=REPO,check=True)
from coffee_detector.af2_ffa import run_af2_ffa_from_start_static_audit
audit=run_af2_ffa_from_start_static_audit(D0,STATIC,device='cuda:0')
print(json.dumps(audit,indent=2))
if audit['decision']!='PASS' or audit['training_authorized'] is not True or audit['test_access_authorized'] is not False:
    raise RuntimeError('STOP: from-start static audit gagal.')
print('STATIC AUTHORIZATION PASS')


In [ ]:
# Train exactly the two frozen Stage-1 arms for this seed.
for ARM in ('AF2FS','AF2FFAB2FS'):
    RESULT=OUT/'val_reports'/f'{ARM}_seed{SEED}_result.json'
    LOG=OUT/f'{ARM}_seed{SEED}.log'
    if not RESULT.is_file():
        cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_from_start_arm',
             '--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),
             '--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUT),
             '--seed',str(SEED),'--device','0','--authorize-training']
        print('START/RESUME',ARM,'seed',SEED,flush=True)
        with LOG.open('a',encoding='utf-8') as stream:
            p=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
        previous=None
        while p.poll() is None:
            csv=OUT/ARM/f'{ARM}_seed{SEED}'/'results.csv'
            epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
            if epoch!=previous: print(f'{ARM} seed {SEED}: {epoch}/50 epoch',flush=True); previous=epoch
            time.sleep(120)
        if p.returncode:
            tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-180:]) if LOG.is_file() else '<no log>'
            raise RuntimeError(f'{ARM} gagal, returncode={p.returncode}\n{tail}')
    else:
        print('REUSE COMPLETE RESULT:',RESULT)
    if not RESULT.is_file(): raise RuntimeError(f'Hasil tidak ditemukan: {RESULT}')
    result=json.loads(RESULT.read_text(encoding='utf-8'))
    assert result['arm']==ARM and result['seed']==SEED and result['evaluation_split']=='val' and result['test_images_accessed'] is False
    print(ARM,{k:result['metrics'][k] for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})


In [ ]:
A=json.loads((OUT/'val_reports'/f'AF2FS_seed{SEED}_result.json').read_text(encoding='utf-8'))
B=json.loads((OUT/'val_reports'/f'AF2FFAB2FS_seed{SEED}_result.json').read_text(encoding='utf-8'))
keys=('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')
print('\n=== PAIRED SEED RESULT ===')
for k in keys: print(k,'AF2=',A['metrics'][k],'FFAB2=',B['metrics'][k],'delta=',B['metrics'][k]-A['metrics'][k])
meta={'branch':BRANCH,'commit':COMMIT,'seed':SEED,'evaluation_split':'val','test_images_accessed':False,'stage1_decision_run':False}
(OUT/'kaggle_run_meta.json').write_text(json.dumps(meta,indent=2)+'\n',encoding='utf-8')
archive=Path(shutil.make_archive(str(WORK/f'af2-ffab2-from-start-seed{SEED}-output'),'zip',OUT))
print('\nFINAL ZIP:',archive)
print('AF2 RESULT:',OUT/'val_reports'/f'AF2FS_seed{SEED}_result.json')
print('FFAB2 RESULT:',OUT/'val_reports'/f'AF2FFAB2FS_seed{SEED}_result.json')
print('STOP HERE. Jalankan seed 42/123/2026 sebagai run terpisah, lalu gunakan notebook Stage-1 Decision. Jangan buka locked test dan jangan jalankan DCT sebelum Stage-1 PASS.')
